# 第105章 用户消费价值预测项目

使用 UCI Online Retail 公开交易数据，按照“问题定义—数据准备—模型训练—模型评价—模型理解”的教学路径，预测客户未来消费金额。

## 项目背景

在统一观察日，根据客户此前的交易行为预测未来窗口内的消费金额。本章关注规范的回归建模过程，不把预测相关性解释为营销措施的因果效果。

## 学习目标

- 理解客户价值预测的样本粒度与时间窗口
- 审计并清洗真实交易数据
- 构造无时间穿越的客户特征与目标
- 比较回归基线、线性模型和树模型
- 使用金额误差、Top-K与错误切片理解模型


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| InvoiceNo | 发票号 | C开头通常为取消单 |
| InvoiceDate | 交易时间 | 划分观察窗口与未来窗口 |
| CustomerID | 客户编号 | 聚合到一位客户一行 |
| Quantity/UnitPrice | 数量/单价 | 构造有效交易金额 |
| future_revenue | 未来消费金额 | 回归目标 |

## 数据质量检查清单

- 重复、取消、退货和非正价格
- CustomerID缺失和清洗保留率
- 观察窗口与目标窗口严格分离
- 客户主键唯一
- 未来零消费比例和金额长尾
- 测试集不参与模型选择


## 项目任务

1. 明确客户粒度、预测时点和未来窗口
2. 审计原始交易数据
3. 清洗交易并记录样本变化
4. 构造历史特征和未来目标
5. 探索目标分布并建立Dummy基线
6. 划分客户训练集与测试集
7. 比较Ridge与梯度提升模型
8. 在原始金额尺度和Top-K上评价
9. 分析错误样本与客户分组
10. 解释特征重要性并总结模型局限


## 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 原始数据质量审计

先量化重复、取消、缺失和异常数值，建立可追溯的数据起点。


In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("/datasets/uci_online_retail_200k.csv", parse_dates=['InvoiceDate'])
audit=pd.Series({'原始行':len(raw), '完全重复':raw.duplicated().sum(), '客户缺失':raw.CustomerID.isna().sum(), '取消单':raw.InvoiceNo.astype(str).str.startswith('C').sum(), '数量非正':(raw.Quantity<=0).sum(), '单价非正':(raw.UnitPrice<=0).sum()})
print(audit.to_string()); print('日期范围:', raw.InvoiceDate.min(), '至', raw.InvoiceDate.max()); display(raw.head())


## 2. 清洗交易并记录样本变化

删除重复后，仅保留有客户编号的有效正向销售，并计算清洗保留率。


In [ ]:
dedup = raw.drop_duplicates().copy()
valid = (~dedup.InvoiceNo.astype(str).str.startswith('C'))&(dedup.Quantity>0)&(dedup.UnitPrice>0)&dedup.CustomerID.notna()
sales = dedup.loc[valid].copy(); sales['revenue']=sales.Quantity*sales.UnitPrice
clean_report = pd.Series({'去重后':len(dedup), '有效销售':len(sales), '删除行':len(raw)-len(sales), '保留率':len(sales)/len(raw)})
print(clean_report.round(3).to_string()); print('单行金额分位数:\n', sales.revenue.quantile([.5,.9,.99,.999]).round(2))


## 3. 定义时间窗口并构造客户样本

截止日前的数据只生成特征，截止日后的数据只生成目标，从源头防止时间穿越。


In [ ]:
cutoff=sales.InvoiceDate.quantile(.8).normalize(); hist=sales[sales.InvoiceDate<cutoff]; future=sales[sales.InvoiceDate>=cutoff]
customer_features = hist.groupby('CustomerID').agg(recency=('InvoiceDate', lambda x:(cutoff-x.max().normalize()).days), frequency=('InvoiceNo', 'nunique'), monetary=('revenue', 'sum'), items=('Quantity', 'sum'), products=('StockCode', 'nunique'), active_days=('InvoiceDate', lambda x:(x.max().normalize()-x.min().normalize()).days+1))
future_target = future.groupby('CustomerID').revenue.sum().rename('future_revenue')
customer = customer_features.join(future_target, how='left').fillna({'future_revenue':0}); assert customer.index.is_unique
print('观察截止日:', cutoff.date(), '客户数:', len(customer), '未来零消费:', f'{(customer.future_revenue==0).mean():.1%}')


## 4. 探索目标分布与客户差异

金额目标同时具有大量零值和明显长尾，后续需要在对数尺度训练、原始金额尺度评价。


In [ ]:
target_summary = customer.future_revenue.describe(percentiles=[.5,.75,.9,.95,.99]).round(2)
customer['history_value_group']=pd.qcut(customer.monetary,4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
group_summary = customer.groupby('history_value_group', observed=True).agg(customers=('future_revenue', 'size'), future_positive_rate=('future_revenue', lambda x:(x>0).mean()), future_mean=('future_revenue', 'mean'), future_median=('future_revenue', 'median'))
print(target_summary); display(group_summary.round(2))


## 5. 客户级划分与Dummy基线

按客户划分训练集和测试集，并用中位数回归器建立最低比较基线。


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split

cols = ['recency', 'frequency', 'monetary', 'items', 'products', 'active_days']; X=customer[cols].clip(lower=0); y_raw=customer.future_revenue; y=np.log1p(y_raw)
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=.25, random_state=105, stratify=(y_raw>0))
dummy = DummyRegressor(strategy='median').fit(X_train, y_train)
print('训练/测试客户:', len(X_train), len(X_test)); print('训练/测试未来有消费比例:', f'{(y_train>0).mean():.1%}', f'{(y_test>0).mean():.1%}')


## 6. 比较线性模型与树模型

使用五折交叉验证比较Ridge与梯度提升，以对数目标MAE选择模型。


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {'Ridge':make_pipeline(StandardScaler(), Ridge(alpha=10)), '梯度提升':HistGradientBoostingRegressor(max_iter=180, max_leaf_nodes=12, l2_regularization=2, random_state=105)}
cv = KFold(5, shuffle=True, random_state=105); rows=[]
for name, model in models.items():
    scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error'); rows.append([name, scores.mean(), scores.std()])
validation = pd.DataFrame(rows, columns=['model', 'CV_log_MAE', 'std']).sort_values('CV_log_MAE'); display(validation.round(3))
best_name = validation.iloc[0].model; best_model=models[best_name].fit(X_train, y_train)


## 7. 在原始金额尺度评价模型

把预测还原为金额，联合报告MAE、RMSE、R²和Dummy基线误差。


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

prediction = np.maximum(0, np.expm1(best_model.predict(X_test))); actual=np.expm1(y_test); baseline=np.maximum(0, np.expm1(dummy.predict(X_test)))
metrics = pd.Series({'MAE_GBP':mean_absolute_error(actual, prediction), 'RMSE_GBP':mean_squared_error(actual, prediction)**.5, 'R2':r2_score(actual, prediction), 'Dummy_MAE':mean_absolute_error(actual, baseline)})
result = X_test.copy(); result['actual']=actual.to_numpy(); result['prediction']=prediction; result['absolute_error']=(result.actual-result.prediction).abs()
print('最佳模型:', best_name); print(metrics.round(2).to_string())


## 8. 使用Top-K检查排序能力

Top-K只作为模型排序评价，比较不同观察比例覆盖了多少真实未来消费金额。


In [ ]:
ranked = result.sort_values('prediction', ascending=False); total_revenue=max(ranked.actual.sum(),1e-9); topk_rows=[]
for share in [.05,.10,.20]:
    n = max(1, int(len(ranked)*share)); topk_rows.append([f'{share:.0%}', n, ranked.head(n).actual.sum()/total_revenue, ranked.head(n).actual.mean()])
topk = pd.DataFrame(topk_rows, columns=['观察比例', '客户数', '真实金额覆盖率', '组内平均金额']); display(topk.round(3))


## 9. 错误切片与高误差案例

分别检查未来零消费/有消费客户，以及不同历史价值组的误差。


In [ ]:
error_table = result.copy(); error_table['future_status']=np.where(error_table.actual>0, '未来有消费', '未来零消费'); error_table['history_quartile']=pd.qcut(error_table.monetary,4, labels=False, duplicates='drop')+1
status_error = error_table.groupby('future_status').absolute_error.agg(['count', 'mean', 'median'])
value_error = error_table.groupby('history_quartile').absolute_error.agg(['count', 'mean', 'median'])
print('按未来状态误差:\n', status_error.round(2)); print('按历史价值分组误差:\n', value_error.round(2)); display(error_table.nlargest(8, 'absolute_error')[['actual', 'prediction', 'absolute_error', 'monetary', 'frequency']].round(2))


## 10. 特征解释与模型局限

置换重要性说明模型依赖哪些历史信号，不能据此断言这些变量会导致未来消费。


In [ ]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(best_model, X_test, y_test, n_repeats=5, scoring='neg_mean_absolute_error', random_state=105)
importance = pd.Series(permutation.importances_mean, index=cols).sort_values(ascending=False)
print('置换重要性:\n', importance.round(4)); print('局限: 固定交易样本、客户编号缺失、未来零值多且金额长尾；预测关系不代表营销干预的因果效果。')


## 结论与表达

- 客户级预测必须先定义观察窗口和未来窗口
- 金额长尾需要区分训练尺度与评价尺度
- 总体误差、Top-K和分组误差回答不同问题
- 特征重要性反映预测依赖而不是因果关系


## 项目验收清单

- 完成原始审计与清洗报告
- 观察和目标窗口无重叠
- 比较Dummy与两个候选模型
- 报告MAE、RMSE、R²和Top-K
- 完成错误切片与置换重要性解释

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Online Retail 公开交易数据，按照“问题定义—数据准备—模型训练—模型评价—模型理解”的教学路径，预测客户未来消费金额。


### 你已经完成

- 理解客户价值预测的样本粒度与时间窗口
- 审计并清洗真实交易数据
- 构造无时间穿越的客户特征与目标
- 比较回归基线、线性模型和树模型
- 使用金额误差、Top-K与错误切片理解模型


### 建模流程速查

| 阶段 | 学习内容 |
| --- | --- |
| 步骤 1 | 明确客户粒度、预测时点和未来窗口 |
| 步骤 2 | 审计原始交易数据 |
| 步骤 3 | 清洗交易并记录样本变化 |
| 步骤 4 | 构造历史特征和未来目标 |
| 步骤 5 | 探索目标分布并建立Dummy基线 |
| 步骤 6 | 划分客户训练集与测试集 |
| 步骤 7 | 比较Ridge与梯度提升模型 |
| 步骤 8 | 在原始金额尺度和Top-K上评价 |
| 步骤 9 | 分析错误样本与客户分组 |
| 步骤 10 | 解释特征重要性并总结模型局限 |


### 质量与结论提醒

- 重复、取消、退货和非正价格
- CustomerID缺失和清洗保留率
- 观察窗口与目标窗口严格分离
- 客户级预测必须先定义观察窗口和未来窗口
- 金额长尾需要区分训练尺度与评价尺度
- 总体误差、Top-K和分组误差回答不同问题
- 特征重要性反映预测依赖而不是因果关系


### 学习检查

- [ ] 完成原始审计与清洗报告
- [ ] 观察和目标窗口无重叠
- [ ] 比较Dummy与两个候选模型
- [ ] 报告MAE、RMSE、R²和Top-K
- [ ] 完成错误切片与置换重要性解释


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
